In [1]:
import os

from io import StringIO
from google.cloud import storage
from dotenv import load_dotenv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, EsmForMaskedLM
from tokenizers import Tokenizer
from peft import get_peft_model, LoraConfig, TaskType

/opt/anaconda3/envs/MachLearn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from plm_compare_progen2 import *
from plm_compare_esm import *
from protein_data import *
from pro_gen2_lora import *

In [15]:
device = 'cpu'
print(f"Using {device} device")
model_name = "hugohrban/progen2-medium"
base_model, tokenizer = initialize_progen2_noeval(model_name)

Using cpu device


In [4]:
filename = '/Users/johnhutchens/Desktop/Practicum/Data/Domainome/dict_domainome_uniprot_new.pkl'
with open(filename, "rb") as f:
    dict_uniprot = pickle.load(f)

In [16]:
dict_uniprot

{'P00519': {'sequence': 'MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAARWNSKENLLAGPSENDPNLFVALYDFVASGDNTLSITKGEKLRVLGYNHNGEWCEAQTKNGQGWVPSNYITPVNSLEKHSWYHGPVSRNAAEYLLSSGINGSFLVRESESSPGQRSISLRYEGRVYHYRINTASDGKLYVSSESRFNTLAELVHHHSTVADGLITTLHYPAPKRNKPTVYGVSPNYDKWEMERTDITMKHKLGGGQYGEVYEGVWKKYSLTVAVKTLKEDTMEVEEFLKEAAVMKEIKHPNLVQLLGVCTREPPFYIITEFMTYGNLLDYLRECNRQEVNAVVLLYMATQISSAMEYLEKKNFIHRDLAARNCLVGENHLVKVADFGLSRLMTGDTYTAHAGAKFPIKWTAPESLAYNKFSIKSDVWAFGVLLWEIATYGMSPYPGIDLSQVYELLEKDYRMERPEGCPEKVYELMRACWQWNPSDRPSFAEIHQAFETMFQESSISDEVEKELGKQGVRGAVSTLLQAPELPTKTRTSRRAAEHRDTTDVPEMPHSKGQGESDPLDHEPAVSPLLPRKERGPPEGGLNEDERLLPKDKKTNLFSALIKKKKKTAPTPPKRSSSFREMDGQPERRGAGEEEGRDISNGALAFTPLDTADPAKSPKPSNGAGVPNGALRESGGSGFRSPHLWKKSSTLTSSRLATGEEEGGGSSSKRFLRSCSASCVPHGAKDTEWRSVTLPRDLQSTGRQFDSSTFGGHKSEKPALPRKRAGENRSDQVTRGTVTPPPRLVKKNEEAADEVFKDIMESSPGSSPPNLTPKPLRRQVTVAPASGLPHKEEAGKGSALGTPAAAEPVTPTSKAGSGAPGGTSKGPAEESRVRRHKHSSESPGRDKGKLSRLKPAPPPPPAASAGKAGGKPSQSPSQEAAGEAVLGAKTKATSLVDAVNSDAAKPSQPGEGLKKPVLPATPKPQSAKPSGTP

In [7]:
filename = '/Users/johnhutchens/Desktop/Practicum/Data/Domainome/dict_dn_fitness.pkl'
with open(filename, "rb") as f:
    dict_dn_fitness = pickle.load(f)

In [ ]:
# load_dotenv()
# cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

# client = storage.Client()
# bucket = client.bucket('domainome-data')
# blob = bucket.blob('SupplementaryTable2.txt')

# df = pd.read_csv(StringIO(blob.download_as_text()), sep='\t')

In [9]:
keys = list(dict_dn_fitness.keys())

In [14]:
i = 124
key = keys[i]

print(key)

dict_dn_fitness[key]

P11308_PF02198_117


{'dom_seq': 'NMTTNERRVIVPADPTLWSTDHVRQWLEWAVKEYGLPDVNILLFQNIDGKELCKMTKDDFQRLTPSYNADILLSHLHYLRE',
 'uniprot_id': 'P11308',
 'pfam': 'PF02198',
 'domain_start': '117',
 'fitness': [-0.143615892377958,
  -0.0573643980783941,
  -0.0710352811299449,
  -0.0379025722279556,
  0.011787123423442,
  -0.154231286654784,
  -0.279181522656227,
  0.0065463201904485,
  -0.113537309675917,
  -0.0250117632357844,
  -0.0866871775842565,
  nan,
  -0.196269103006686,
  -0.0106272316451942,
  -0.0538046714755485,
  -0.0969983770422937,
  0.0019902107825001,
  -0.202754958941441,
  -0.167679133247654,
  0.0291165197491137,
  -0.295821796788831,
  -0.165643702014768,
  -0.195069611399005,
  -0.212695878704043,
  -0.0484173914541125,
  -0.337419246005247,
  -0.0456292228182951,
  -0.14271199433579,
  -0.143528943075051,
  -0.0419289806916528,
  nan,
  -0.0276892226710349,
  0.0042180205416016,
  0.0370952466730518,
  -0.048936021478307,
  -0.105478701805724,
  -0.0327826975063506,
  -0.416561506409941,
  -0.1

In [17]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)

i = 124
domain_id = keys[i]
print(key)
dict_entry = dict_dn_fitness[domain_id]

dom_pos = int(dict_entry['domain_start'])
dom_seq = dict_entry['dom_seq']

uniprot_id = dict_entry['uniprot_id']
protein_seq = dict_uniprot[uniprot_id]['sequence']

fitness_list = dict_entry['fitness']
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor

loss = listwise_ranking_loss

eps = 5
lr = 1e-3
num_samples = 4

model, train_losses, val_losses = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                        lora_config, protein_seq, dom_seq, dom_pos,
                                        exp_tensor, loss, lrate=lr, num_epochs=eps, 
                                        k=0.8, num_samples=i, print_info=False)

print(f"Training loss = {train_losses}")
print(f"Validation loss = {val_losses}")


    

P11308_PF02198_117
Training loss = [10.483512878417969, 6.313688278198242, 6.508936405181885, 4.549412727355957, 4.263123989105225]
Validation loss = [13.404886245727539, 6.034444332122803, 4.742634296417236, 4.9106926918029785, 4.196688175201416]


In [ ]:
dict_LLRs = {}

In [ ]:
# dict_uniprot_LLRs = dict_uniprot.copy()

adding full LLRs for uniprot ids LoRA model

In [22]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): ProGenForCausalLM(
      (transformer): ProGenModel(
        (wte): Embedding(32, 1536)
        (drop): Dropout(p=0.0, inplace=False)
        (h): ModuleList(
          (0-26): 27 x ProGenBlock(
            (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
            (attn): ProGenAttention(
              (attn_dropout): Dropout(p=0.0, inplace=False)
              (resid_dropout): Dropout(p=0.0, inplace=False)
              (qkv_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=4608, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4608, bias=False)
      

In [24]:
dict_uniprot_LLRs

{'P00519': {'sequence': 'MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAARWNSKENLLAGPSENDPNLFVALYDFVASGDNTLSITKGEKLRVLGYNHNGEWCEAQTKNGQGWVPSNYITPVNSLEKHSWYHGPVSRNAAEYLLSSGINGSFLVRESESSPGQRSISLRYEGRVYHYRINTASDGKLYVSSESRFNTLAELVHHHSTVADGLITTLHYPAPKRNKPTVYGVSPNYDKWEMERTDITMKHKLGGGQYGEVYEGVWKKYSLTVAVKTLKEDTMEVEEFLKEAAVMKEIKHPNLVQLLGVCTREPPFYIITEFMTYGNLLDYLRECNRQEVNAVVLLYMATQISSAMEYLEKKNFIHRDLAARNCLVGENHLVKVADFGLSRLMTGDTYTAHAGAKFPIKWTAPESLAYNKFSIKSDVWAFGVLLWEIATYGMSPYPGIDLSQVYELLEKDYRMERPEGCPEKVYELMRACWQWNPSDRPSFAEIHQAFETMFQESSISDEVEKELGKQGVRGAVSTLLQAPELPTKTRTSRRAAEHRDTTDVPEMPHSKGQGESDPLDHEPAVSPLLPRKERGPPEGGLNEDERLLPKDKKTNLFSALIKKKKKTAPTPPKRSSSFREMDGQPERRGAGEEEGRDISNGALAFTPLDTADPAKSPKPSNGAGVPNGALRESGGSGFRSPHLWKKSSTLTSSRLATGEEEGGGSSSKRFLRSCSASCVPHGAKDTEWRSVTLPRDLQSTGRQFDSSTFGGHKSEKPALPRKRAGENRSDQVTRGTVTPPPRLVKKNEEAADEVFKDIMESSPGSSPPNLTPKPLRRQVTVAPASGLPHKEEAGKGSALGTPAAAEPVTPTSKAGSGAPGGTSKGPAEESRVRRHKHSSESPGRDKGKLSRLKPAPPPPPAASAGKAGGKPSQSPSQEAAGEAVLGAKTKATSLVDAVNSDAAKPSQPGEGLKKPVLPATPKPQSAKPSGTP

In [29]:
for key in dict_uniprot_LLRs.keys():
    seq = dict_uniprot_LLRs[key]['sequence']
    print(len(seq))
    if len(seq) > 1024:
        seq = seq[:1023]
    lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
    dict_uniprot_LLRs[key]['LoRA_model_LLR'] = llr
    

1130
683
1102
1196
326
343
411
920
1881
3957
4377
871
1248
606
1705
1835
346
562
587
195
3477
475
892
819
575
447
2168
505
3046
766
1362
659
220
196
1086
906
191
1150
786
639
2000
1912
1954
2997
219
79
1034
423
589
215
196
252
205
1661
174
175
174
174
299
178
798
926
70
66
67
443
2517
1505
1486
711
2240
358
531
817
724
382
760
340
348
309
554
621
198
494
1577
82
1226
743
695
716
371
265
83
2414
822
1412
479
349
433
441
208
375
927
403
690
475
323
279
452
2602
632
1017
1322
644
711
572
537
681
454
777
330
422
1106
1586
1580
775
217
1128
797
257
304
207
509
526
2610
1572
4834
185
347
856
747
463
825
510
4374
301
84
462
903
68
620
1721
1697
492
381
852
822
1336
1560
1317
512
3969
676
1130
509
433
727
341
868
463
1132
800
221
434
1491
1455
1481
415
875
984
491
490
486
659
651
1006
482
416
448
2070
585
931
303
267
594
508
228
640
938
1108
828
138
390
526
2514
1098
8525
975
1319
605
732
579
598
482
428
501
289
275
680
593
636
444
486
424
1205
1356
422
505
1689
356
5142
352
284
163
291
942
12

adding full LLRs for uniprot ids base model

In [31]:
device = 'cpu'
print(f"Using {device} device")
model_name = "hugohrban/progen2-medium"
base_model, tokenizer = initialize_progen2_noeval(model_name)

Using cpu device


In [32]:
base_model.eval()

ProGenForCausalLM(
  (transformer): ProGenModel(
    (wte): Embedding(32, 1536)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-26): 27 x ProGenBlock(
        (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
        (attn): ProGenAttention(
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
          (qkv_proj): Linear(in_features=1536, out_features=4608, bias=False)
          (out_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): ProGenMLP(
          (fc_in): Linear(in_features=1536, out_features=6144, bias=True)
          (fc_out): Linear(in_features=6144, out_features=1536, bias=True)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1536, out_features=32, bias=True)
)

In [34]:
for key in dict_uniprot_LLRs.keys():
    seq = dict_uniprot_LLRs[key]['sequence']
    # print(len(seq))
    if len(seq) > 1024:
        seq = seq[:1023]
    lp, rlp, llr = collect_log_prob_pg2(seq, base_model, tokenizer)
    dict_uniprot_LLRs[key]['base_model_LLR'] = llr

In [36]:
path = '/Users/johnhutchens/Desktop/Practicum/Data/Domainome/'
with open(path+"dict_test_LLRs.pkl", "wb") as f:
    pickle.dump(dict_uniprot_LLRs, f)

In [35]:
dict_uniprot_LLRs

{'P00519': {'sequence': 'MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAARWNSKENLLAGPSENDPNLFVALYDFVASGDNTLSITKGEKLRVLGYNHNGEWCEAQTKNGQGWVPSNYITPVNSLEKHSWYHGPVSRNAAEYLLSSGINGSFLVRESESSPGQRSISLRYEGRVYHYRINTASDGKLYVSSESRFNTLAELVHHHSTVADGLITTLHYPAPKRNKPTVYGVSPNYDKWEMERTDITMKHKLGGGQYGEVYEGVWKKYSLTVAVKTLKEDTMEVEEFLKEAAVMKEIKHPNLVQLLGVCTREPPFYIITEFMTYGNLLDYLRECNRQEVNAVVLLYMATQISSAMEYLEKKNFIHRDLAARNCLVGENHLVKVADFGLSRLMTGDTYTAHAGAKFPIKWTAPESLAYNKFSIKSDVWAFGVLLWEIATYGMSPYPGIDLSQVYELLEKDYRMERPEGCPEKVYELMRACWQWNPSDRPSFAEIHQAFETMFQESSISDEVEKELGKQGVRGAVSTLLQAPELPTKTRTSRRAAEHRDTTDVPEMPHSKGQGESDPLDHEPAVSPLLPRKERGPPEGGLNEDERLLPKDKKTNLFSALIKKKKKTAPTPPKRSSSFREMDGQPERRGAGEEEGRDISNGALAFTPLDTADPAKSPKPSNGAGVPNGALRESGGSGFRSPHLWKKSSTLTSSRLATGEEEGGGSSSKRFLRSCSASCVPHGAKDTEWRSVTLPRDLQSTGRQFDSSTFGGHKSEKPALPRKRAGENRSDQVTRGTVTPPPRLVKKNEEAADEVFKDIMESSPGSSPPNLTPKPLRRQVTVAPASGLPHKEEAGKGSALGTPAAAEPVTPTSKAGSGAPGGTSKGPAEESRVRRHKHSSESPGRDKGKLSRLKPAPPPPPAASAGKAGGKPSQSPSQEAAGEAVLGAKTKATSLVDAVNSDAAKPSQPGEGLKKPVLPATPKPQSAKPSGTP